# Noise in Data Labels

![label_noise_intro.png](https://live.staticflickr.com/65535/54328952408_f7e5bcb7c9_z.jpg)

*Image generated using a generative model from OpenArt.ai.*

## Introduction

Data is fundamental in machine learning. Everything starts with it. In practice, however, data is often imperfect and contains some noise that reduces its quality. One type of such noise is **label noise**, meaning that for some observations, the labels are incorrect.

The causes of label noise can vary. It often stems from subjectivity in evaluation—different experts may have differing opinions, for example when assessing emotions in images or grading essays. Another source of errors can be annotator fatigue, which affects concentration and accuracy.

Ambiguities can also arise from poor data quality, making clear classification difficult (e.g., a blurry image of a dog that resembles a wolf). Sometimes labels are generated automatically by artificial intelligence models, which can also make mistakes.

It is also worth mentioning samples that lie on the boundary between classes. Such cases, for instance in medical data where symptoms overlap across diseases, also make it difficult to assign a clear label.

Noisy data makes it harder to train a high-quality model, as the model may focus more on incorrect information than on the underlying general patterns.

## Task

Your task is to train **two** neural networks for correct binary image classification despite partial label noise in the training data. The training set is imbalanced (take this into account in your solution). The validation and test sets (which will be used to evaluate your final solution) contain only correct labels (no noise).

**The model architecture is predefined and cannot be changed.**

Think about why we use two models instead of one (this is a bit of a puzzle) — understanding this will help you grasp and solve the task.

Your role in this assignment is to implement the function `your_selected_indices(targets, losses)`, which selects indices of training samples to be used for training the models. The function takes as input a tensor of data labels (`targets`) and a tensor of loss values from both models (`losses`). The output should be a two-element list, where each element is a tensor containing indices selected for training. One model receives one set of indices, and the other model receives the second set.

Below in the notebook, you will find a cell where you should implement your function. The cell you need to modify is clearly marked. To better understand its purpose, it is worth examining the context and where this function is called within the training loop.

### Evaluation Criteria

The final evaluation will be based on the average balanced accuracy (*BAC*) of the two models:

$$
{BAC}_{mean} = \frac{BAC_1 + BAC_2}{2}
$$

where $BAC_i$ is the balanced accuracy of model $i$, $(i = 1, 2)$.

You can earn between 0 and 100 points for this task.

Your final score will be calculated using the following formula (higher is better), with rounding to integers:

$$
\mathrm{Points} =
\begin{cases}
0 & \text{if } {BAC}*{mean} \leq 0.5 \\
100 \times \frac{{BAC}*{mean} - 0.5}{0.8 - 0.5} & \text{if } 0.5 < {BAC}*{mean} < 0.8 \\
100 & \text{if } {BAC}*{mean} \geq 0.8
\end{cases}
$$

**Note:** To achieve the maximum number of points, it is not necessary to reach the maximum possible balanced accuracy of 1. If ${BAC}_{mean} \geq 0.8$, you will receive the full score.

All evaluation criteria and functions mentioned above are already implemented below.

## Constraints

* Your solution will be tested on the Competition Platform without internet access and in a GPU-enabled environment.
* Evaluation of your final solution on the platform must not exceed **5 minutes** using a GPU.
* You **cannot** modify the model architecture — it must remain the predefined `SmallMobileNet`.

## Submission Files

This notebook completed with your solution (see the `your_selected_indices` function).

## Evaluation

Remember that during evaluation, the flag `FINAL_EVALUATION_MODE` will be set to `True`.

You can earn between 0 and 100 points for this task. The number of points will be calculated on a (hidden) test set on the Competition Platform based on the previously described formula and rounded to an integer. If your solution does not meet the above criteria or fails to execute correctly, you will receive 0 points.

# Starter Code

In this section, we initialize the environment by importing the necessary libraries and functions. The provided code will help you efficiently work with the data and build the appropriate solution.

In [ ]:
# During evaluation of your solution, the value of the FINAL_EVALUATION_MODE flag will be set to True
FINAL_EVALUATION_MODE = False

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################
import os
from tqdm import tqdm
from typing import Optional, Tuple, List

import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader

import torchvision.transforms as transforms
from torchvision.datasets.folder import VisionDataset

from sklearn.metrics import balanced_accuracy_score

## Helper Functions and Constants

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################
SEED = 123
IMAGES_DIR = "data"
TASK_DATASET_LABELS_FILE = "dataset_labels.csv"

ROOT_DIR = os.getcwd()
TRAIN_DATASET_PATH = os.path.join(ROOT_DIR,'train')
VAL_DATASET_PATH = os.path.join(ROOT_DIR, 'val')

TRAIN_DATASET_URL = "1qmNNmDv-wUcAv5mvO6vYJV3mQ2SNIGnI"
VAL_DATASET_URL = "1YUJYD12NmKRSzFJGMrX-a61d6mnTaWbG"

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LEARNING_RATE = 1e-2
NUM_EPOCHS = 6
NUM_CLASSES = 2
BATCH_SIZE = 128
WEIGHT_DECAY = 1e-3

if not FINAL_EVALUATION_MODE:
  print(f"Using {DEVICE} device")

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################
def seed_everything(seed: int) -> None:
    """
    Sets the random seed for reproducibility in Python, NumPy, and PyTorch.

    This function sets the seed for Python, NumPy, and PyTorch random number generators,
    and configures PyTorch to operate in deterministic mode.

    Parameters:
        seed (int): The seed value to set.
    """
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## Data Loading

Using the code below, the data will be loaded and properly prepared.

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################
def download_data(dataset_path, dataset_url):
    """Downloads the Olympiad dataset from Google Drive and saves it in the folder."""
    import gdown
    import shutil

    # Create or reset folder
    output = dataset_path+".zip"
    if os.path.exists(dataset_path):
        shutil.rmtree(dataset_path)
    if os.path.exists(output):
        os.remove(output)

    url = f'https://drive.google.com/uc?id={dataset_url}'
    gdown.download(url, output, fuzzy=True)

    print(f"Downloaded: {output}")

# Download data only if not in FINAL_EVALUATION_MODE
if not FINAL_EVALUATION_MODE:
    download_data(TRAIN_DATASET_PATH, TRAIN_DATASET_URL)
    download_data(VAL_DATASET_PATH, VAL_DATASET_URL)

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################

# Dataset class
class TaskDataset(VisionDataset):
    def __init__(
        self,
        root: str,
        transform: Optional[callable] = None,
    ):
        super().__init__(
            root,
            transform=transform,
        )
        self.root = root

        if not self._check_integrity():
            raise RuntimeError(
                f"Dataset not found. Please check whether the path {self.root} exists. It should contain the folder '{IMAGES_DIR}' and the file '{TASK_DATASET_LABELS_FILE}'"
            )
        self.labels_df = self._read_labels_from_file()
        self.labels_header = 'label'

    def _read_labels_from_file(self) -> pd.DataFrame:
        df = pd.read_csv(os.path.join(self.root, TASK_DATASET_LABELS_FILE))
        return df

    def _check_integrity(self) -> bool:
        return os.path.exists(os.path.join(self.root, IMAGES_DIR)) and os.path.exists(
            os.path.join(self.root, TASK_DATASET_LABELS_FILE)
        )

    def __len__(self) -> int:
        return len(self.labels_df)

    def __getitem__(self, idx: int) -> Tuple[Image.Image, np.ndarray]:
        img = self._load_image(idx)
        label = self._load_label(idx)
        if self.transform is not None:
            img = self.transform(img)
        return img, label

    def _load_image(self, idx: int) -> Image.Image:
        img_path = os.path.join(
            self.root, IMAGES_DIR, self.labels_df.iloc[idx]['file_name']
        )
        img = Image.open(img_path)
        return img

    def _load_label(self, idx: int):
        label = self.labels_df.iloc[idx][self.labels_header]
        return np.array([int(label)])

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################
def unpack_data(unpack_path, dataset_name) -> None:
    dataset_zip_path = os.path.join(ROOT_DIR, dataset_name+".zip")
    dataset_local_dir = os.path.join(unpack_path, dataset_name)
    if not os.path.exists(dataset_local_dir):
        if not os.path.exists(dataset_zip_path):
            raise FileNotFoundError(
                f"File {dataset_zip_path} not found in the current directory."
            )

        with zipfile.ZipFile(dataset_zip_path, "r") as zip_ref:
            zip_ref.extractall(unpack_path)

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################
# Function for loading training and validation data
def load_data() -> Tuple[DataLoader, DataLoader]:
    """
    Function for loading training and validation data using the TaskDataset class.

    The function creates datasets for training and validation data,
    applies a basic transformation (conversion to tensor), and then
    wraps them into DataLoader objects.

    Returns:
        Tuple[DataLoader, DataLoader]: DataLoader objects for the training and validation sets.
    """
    base_transform = transforms.Compose([transforms.ToTensor()])

    train_dataset = TaskDataset(root=TRAIN_DATASET_PATH, transform=base_transform)
    val_dataset = TaskDataset(root=VAL_DATASET_PATH, transform=base_transform)

    train_loader = DataLoader(
        dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=False
    )
    val_loader = DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    return train_loader, val_loader

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################
# Let’s unzip and load the data
if not FINAL_EVALUATION_MODE:
    unpack_data(ROOT_DIR, "train")
    unpack_data(ROOT_DIR, "val")
    train_loader, val_loader = load_data()

## Model Architecture

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################
class SmallMobileNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super(SmallMobileNet, self).__init__()

        # Main convolutional blocks
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU6(inplace=True),
            nn.Conv2d(
                32, 32, kernel_size=3, stride=1, padding=1, groups=32, bias=False
            ),
            nn.BatchNorm2d(32),
            nn.ReLU6(inplace=True),
            nn.Conv2d(32, 64, kernel_size=1, stride=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU6(inplace=True),
            nn.Conv2d(
                64, 64, kernel_size=3, stride=2, padding=1, groups=64, bias=False
            ),
            nn.BatchNorm2d(64),
            nn.ReLU6(inplace=True),
            nn.Conv2d(64, 128, kernel_size=1, stride=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU6(inplace=True),
            nn.Conv2d(
                128, 128, kernel_size=3, stride=2, padding=1, groups=128, bias=False
            ),
            nn.BatchNorm2d(128),
            nn.ReLU6(inplace=True),
            nn.Conv2d(128, 256, kernel_size=1, stride=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU6(inplace=True),
        )

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU6(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

## Evaluation Metric Code

Code similar to the one below will be used to evaluate the solution on the test set.

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################
def predict_and_evaluate(model, val_loader, device, verbose=False):
    model.eval()
    all_preds, all_targets = [], []

    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())

    balanced_accuracy = balanced_accuracy_score(all_targets, all_preds)

    if verbose:
        print(f"Balanced Accuracy: {balanced_accuracy}")

    return balanced_accuracy

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################
def performance(bac_1: float, bac_2: float) -> None:
    """
    Computes and prints the performance score based on two balanced accuracy values.

    The final score is the average of both values, rescaled between predefined bounds,
    which is then converted into a number of earned points.

    Parameters:
        bac_1 (float): Balanced accuracy value for the first model.
        bac_2 (float): Balanced accuracy value for the second model.
    """
    bac_mean = (bac_1 + bac_2) / 2
    if bac_mean <= 0.5:
        points = 0
    elif 0.5 < bac_mean < 0.8:
        points = (bac_mean - 0.5) / (0.8 - 0.5) * 100
        points = int(round(points))
    else:
        points = 100

    print(
        f"Your solution has a mean balanced accuracy of {round(bac_mean, 5)} on the validation set, which gives {points}/100 points."
    )
    return points

## Training Model

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################


# Function for training the model
def train(
    model1,
    model2,
    optimizer1,
    optimizer2,
    criterion,
    train_loader,
    val_loader,
    num_epochs,
    device,
    select_indices_fn,
):

    verbose = False if FINAL_EVALUATION_MODE else True

    # Metric history for each model
    metrics = {
        k: [[], []]
        for k in [
            "train_loss",
            "val_loss",
            "train_bac",
            "val_bac",
        ]
    }
    epochs_range = np.arange(num_epochs) + 1

    # Main training loop
    for epoch in epochs_range:
        print(f"Epoch {epoch}")

        # Statistics history for each model
        stats = {
            k: [0, 0] for k in ["train_loss", "train_total", "val_loss", "val_total"]
        }
        preds_targets = {
            k: [[], []]
            for k in ["train_preds", "train_targets", "val_preds", "val_targets"]
        }

        model1.train(), model2.train()
        for inputs, targets in tqdm(train_loader, desc=f"Epoch {epoch}/{num_epochs}"):
            inputs, targets = inputs.to(device), targets.squeeze().long().to(device)

            outputs = [m(inputs) for m in (model1, model2)]
            losses = [criterion(out, targets) for out in outputs]

            # --- MAIN TASK POINT ---
            selected_indices = select_indices_fn(targets, losses)
            # ---------------------------

            # Backpropagation for each model
            for i, (model, optim) in enumerate(
                [(model1, optimizer1), (model2, optimizer2)]
            ):
                optim.zero_grad()
                sel_idx = selected_indices[i]
                loss = criterion(model(inputs[sel_idx]), targets[sel_idx]).mean()

                loss.backward()
                optim.step()

                # Statistics history
                stats["train_loss"][i] += loss.item() * len(sel_idx)
                stats["train_total"][i] += len(sel_idx)
                preds = outputs[i].max(1)[1]
                preds_targets["train_preds"][i].extend(preds[sel_idx].cpu().numpy())
                preds_targets["train_targets"][i].extend(targets[sel_idx].cpu().numpy())

        # Evaluation on validation set
        if verbose:
            model1.eval(), model2.eval()
            with torch.no_grad():
                for inputs, targets in tqdm(
                    val_loader, desc=f"Validation {epoch}/{num_epochs}"
                ):
                    inputs, targets = inputs.to(device), targets.squeeze().long().to(
                        device
                    )

                    for i, model in enumerate([model1, model2]):
                        outputs = model(inputs)
                        loss = criterion(outputs, targets).mean()
                        preds = outputs.max(1)[1]

                        stats["val_loss"][i] += loss.item() * inputs.size(0)
                        stats["val_total"][i] += inputs.size(0)
                        preds = outputs.max(1)[1]
                        preds_targets["val_preds"][i].extend(preds.cpu().numpy())
                        preds_targets["val_targets"][i].extend(targets.cpu().numpy())

        # Metric computation
        if verbose:
            models = [model1, model2]
            for i in range(2):
                for phase in ["train", "val"]:
                    preds = preds_targets[f"{phase}_preds"][i]
                    targets = preds_targets[f"{phase}_targets"][i]

                    metrics[f"{phase}_loss"][i].append(
                        stats[f"{phase}_loss"][i] / stats[f"{phase}_total"][i]
                    )
                    metrics[f"{phase}_bac"][i].append(
                        balanced_accuracy_score(targets, preds)
                    )

                print(
                    f"Model{i+1} - Train Loss: {metrics['train_loss'][i][-1]:.4f}, "
                    f"Train balanced accuracy: {metrics['train_bac'][i][-1]:.4f} --- "
                    f"Validation Loss: {metrics['val_loss'][i][-1]:.4f}, "
                    f"Validation balanced accuracy: {metrics['val_bac'][i][-1]:.4f}, "
                )

    # Plot generation
    if verbose:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
        colors = ["#fa2729", "#ac1a1c", "#1a6aff", "#144aad"]
        linestyles = ["-", "--"]

        for i, model_name in enumerate(["Model1", "Model2"]):
            for j, phase in enumerate(["train", "val"]):

                color = colors[i * 2 + j]
                ax1.plot(
                    epochs_range,
                    metrics[f"{phase}_loss"][i],
                    color=color,
                    marker="o",
                    linestyle=linestyles[j],
                    label=f"{model_name} {phase.title()} Loss",
                )
                ax2.plot(
                    epochs_range,
                    metrics[f"{phase}_bac"][i],
                    color=color,
                    marker="o",
                    linestyle=linestyles[j],
                    label=f"{model_name} {phase.title()} balanced accuracy",
                )

        for ax, title in zip([ax1, ax2], ["Loss", "Balanced accuracy"]):
            ax.set_title(f"Training and Validation {title}")
            ax.set_xticks(epochs_range)
            ax.set_xlabel("Epochs")
            ax.set_ylabel(title)
            ax.legend()

        plt.tight_layout()
        plt.show()

## Example Solution

Below we present a simplified solution that serves as an example demonstrating the basic functionality of the notebook. It can be used as a starting point for developing your own solution.

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################
def default_select_indices(targets, losses):
    # All indices for both models
    selected_indices = [torch.arange(targets.shape[0]).to(DEVICE) for _ in range(2)]
    return selected_indices

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################
# IMPORTANT: We always train two models and evaluate them

if not FINAL_EVALUATION_MODE:
    seed_everything(SEED)
    criterion = nn.CrossEntropyLoss(reduction="none")

    model1 = SmallMobileNet(NUM_CLASSES).to(DEVICE)
    model2 = SmallMobileNet(NUM_CLASSES).to(DEVICE)

    optimizer1 = AdamW(model1.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    optimizer2 = AdamW(model2.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    seed_everything(SEED)
    train(
        model1,
        model2,
        optimizer1,
        optimizer2,
        criterion,
        train_loader,
        val_loader,
        NUM_EPOCHS,
        DEVICE,
        select_indices_fn=default_select_indices,
    )

    # Evaluation of the example solution
    bac_1 = predict_and_evaluate(model1, val_loader, DEVICE, verbose=True)
    bac_2 = predict_and_evaluate(model2, val_loader, DEVICE, verbose=True)

    performance(bac_1, bac_2)

# Your Solution

In this section you should place your solution. Make changes **only here**!

The current starting point is this example solution. Your task is to modify only the body of the function. In your solution, do not use `default_select_indices`.


In [ ]:
def your_select_indices(
    targets: torch.Tensor, losses: List[torch.Tensor]
) -> List[torch.Tensor]:
    """
    Function selecting training indices to be used for training the models.

    Parameters:
        targets: Training labels for the current batch as a tensor. Shape: (batch_size,)
        losses: A two-element list containing tensors of loss values for each model. Shape: [(batch_size,), (batch_size,)]

    Returns:
        A list of indices selected for updating model weights.
    """
    # Your code
    selected_indices = []
    return default_select_indices(targets, losses)

# Evaluation

Running the cells below will allow you to check how many points your solution would achieve on the validation data. Before submission, make sure that the entire notebook (including with the `FINAL_EVALUATION_MODE = True` flag set) executes from start to finish without errors and without requiring any user intervention after selecting “Run All”.


In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################
def final_evaluate(evaluate_data_path, model1, model2):

    base_transform = transforms.Compose([transforms.ToTensor()])
    evaluate_dataset = TaskDataset(root=evaluate_data_path, transform=base_transform)
    evaluate_loader = DataLoader(
        dataset=evaluate_dataset, batch_size=BATCH_SIZE, shuffle=False
    )

    bac_1 = predict_and_evaluate(model1, evaluate_loader, DEVICE, verbose=True)
    bac_2 = predict_and_evaluate(model2, evaluate_loader, DEVICE, verbose=True)
    return performance(bac_1, bac_2)

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################
if not FINAL_EVALUATION_MODE:
    seed_everything(SEED)
    criterion = nn.CrossEntropyLoss(reduction="none")

    model1 = SmallMobileNet(NUM_CLASSES).to(DEVICE)
    model2 = SmallMobileNet(NUM_CLASSES).to(DEVICE)

    optimizer1 = AdamW(model1.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    optimizer2 = AdamW(model2.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    seed_everything(SEED)
    train(
        model1,
        model2,
        optimizer1,
        optimizer2,
        criterion,
        train_loader,
        val_loader,
        NUM_EPOCHS,
        DEVICE,
        select_indices_fn=your_select_indices,
    )

    final_evaluate(VAL_DATASET_PATH, model1, model2)

Your function `your_select_indices` will be saved to a file named `your_select_indices.pkl`, and then used to train the models on the training set (according to the code above). The final score will be computed based on classification performance on the test set.

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################
if FINAL_EVALUATION_MODE:
    import cloudpickle

    OUTPUT_PATH = "file_output"
    FUNCTION_FILENAME = "your_select_indices.pkl"
    FUNCTION_OUTPUT_PATH = os.path.join(OUTPUT_PATH, FUNCTION_FILENAME)

    if not os.path.exists(OUTPUT_PATH):
        os.makedirs(OUTPUT_PATH)

    with open(FUNCTION_OUTPUT_PATH, "wb") as f:
        cloudpickle.dump(your_select_indices, f)